In [3]:
import os
from transformers import AutoTokenizer, LlamaForCausalLM
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.getenv("HF_TOKEN"))
model = LlamaForCausalLM.from_pretrained(MODEL_NAME, token=os.getenv("HF_TOKEN"))


In [75]:
import pandas as pd

results = []
df = pd.read_csv('../WikiTableQuestions/data/training-before300.tsv', sep='\t')
for index, row in df.iterrows():
    try:
        context_file = row['context']
        # try:
        contxet_data = pd.read_csv('../WikiTableQuestions/' + context_file)
        file_content = contxet_data.to_string(index=False,col_space=12)

        raw_prompt = [
            {"role": "system", "content": "You are a helpful assistant who can read tables in the context field and answer questions about the table. You only output the answer with minimal words, no explanation, no dialogue, and no complete sentence."},
            {"role": "user", "content": f"question: {row['utterance']}, \n context:\n{file_content}"}
        ]
        formatted_prompt = tokenizer.apply_chat_template(raw_prompt, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted_prompt, return_tensors="pt", padding=True)

        attention_mask = inputs["attention_mask"]

        outputs = model.generate(
            inputs['input_ids'], 
            attention_mask=attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=500
        )
        raw_answer = tokenizer.decode(outputs[0], skip_special_tokens=False)
        # Extract the assistant's answer
        start = raw_answer.find('<|start_header_id|>assistant<|end_header_id|>') + len('<|start_header_id|>assistant<|end_header_id|>')
        end = raw_answer.find('<|eot_id|>', start)
        answer = raw_answer[start:end].strip()
        
        #store the answer to results
        result = (index, answer, row['targetValue'])
        print(result)
        results.append(result)
        if index > 20:
            break

    except Exception as e:
        print(f"Error reading context file {context_file}: {e}")


(0, '2005', '2004')
(1, 'Tampere, Finland', 'Bangkok, Thailand')
(2, 'Ballymore Eustace', 'Wolfe Tones')
(3, '3,549', '12,467')
(4, 'Opponent: Chelsea', 'Derby County')
(5, '7', '4')
(6, 'Varbergs GIF (D3)', 'Varbergs GIF')
(7, 'Lake Palas Tuzla  106 km2\nLake Tuz  1500 km2', 'Lake Palas Tuzla')
(8, '* 4000', '32')
(9, '1. Robert Täht\n2. Siim Ennemuist\n3. Edgar Järvekülg\n4. Jaanus Nõmmsalu\n5. Andri Aganits', 'Siim Ennemuist|Andri Aganits')
(10, '2', '6')
(11, '2002 Asian Championships', 'Bangkok, Thailand')
(12, 'The number of temples in Imabari and Matsuyama is 16.', '2')
(13, '2005', '1999-2000')
(14, 'Marina Kilius / Hans-Jürgen Bäumler', 'Kim Yu-na')
(15, '15th', 'New Delhi, India')
(16, '55', '58')
(17, '1. Albert', 'Abbott')
(18, '1\n2\n1\n1\n1\n2\n1\n2\n1\n1\n2\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1', '20')
(19, '1', '3')
(20, '1991 OK Alright A Huh O Yeah', 'The Sound Of Trees')
(21, 'M1038A1', 'KM-45 Series')


In [76]:
total, correct = 0, 0
for index, answer, target in results:
    total += 1
    if answer == target:
        correct += 1

print(f"Accuracy: {correct / total}")


Accuracy: 0.0
